Modeling: Parallel Bug Fix
==========================

Depending on the operating system (e.g. Linux, Mac, Windows) and Python version, running a Python script or Jupyter
notebook may lead to a error being raised when the search begins.

The root cause of this error is that Python parallelization and JAX may only work when the script is run in a
particular format, which this script illustrates.

The code in this script is identical to the `autogalaxy_workspace/scripts/imaging/modeling.py` script.
Comments have therefore been removed to avoid repetition and make the script more concise.

__Contents__

- **The Fix:** The code pattern that fixes parallelization issues on certain operating systems.
- **Trouble Shooting:** What to do if parallelization still does not work after applying the fix.

__The Fix__

The fix which makes parallelization work is at the end of the script, where we use the following code:

`if __name__ == "__main__":`

    `fit()`

The reason this fixes parallelization is beyond the scope of this tutoriag. However, if you are curious, a quick
Google search will provide you with a detailed explanation! For example, the stack overflow page below has
some good answers:

 https://stackoverflow.com/questions/20360686/compulsory-usage-of-if-name-main-in-windows-while-using-multiprocessi

This fix will work for all dataset formats (e.g. `imaging`, `interferometer`) and should therefore be adopted
for any modeling script you write that has the error described above.

__Trouble Shooting__

If you still cannot get parallelization to work, please ask to be added to the SLACK
channel (by emailing me https://github.com/Jammy2211), where we will be able to provide support.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:


def fit():
    from autogalaxy import (
        jax_wrapper,
    )  # Ensures JAX environment variables are set before other imports

    from autogalaxy import setup_notebook; setup_notebook()

    from pathlib import Path
    import autofit as af
    import autogalaxy as ag
    import autogalaxy.plot as aplt

    """
    __Dataset__
    """
    dataset_name = "simple"
    dataset_path = Path("dataset") / "imaging" / dataset_name

    """
    __Dataset Auto-Simulation__

    If the dataset does not already exist on your system, it will be created by running the corresponding
    simulator script. This ensures that all example scripts can be run without manually simulating data first.
    """
    if ag.util.dataset.should_simulate(str(dataset_path)):
        import subprocess
        import sys

        subprocess.run(
            [sys.executable, "scripts/imaging/simulator.py"],
            check=True,
        )

    dataset = ag.Imaging.from_fits(
        data_path=Path(dataset_path) / "data.fits",
        psf_path=Path(dataset_path) / "psf.fits",
        noise_map_path=Path(dataset_path) / "noise_map.fits",
        pixel_scales=0.1,
    )

    aplt.subplot_imaging_dataset(dataset=dataset)

    """
    __Mask__
    """
    mask_radius = 3.0

    mask = ag.Mask2D.circular(
        shape_native=dataset.shape_native,
        pixel_scales=dataset.pixel_scales,
        radius=mask_radius,
    )

    dataset = dataset.apply_mask(mask=mask)

    aplt.subplot_imaging_dataset(dataset=dataset)

    """
    __Over Sampling__
    """
    over_sample_size = ag.util.over_sample.over_sample_size_via_radial_bins_from(
        grid=dataset.grid,
        sub_size_list=[4, 2, 2],
        radial_list=[0.3, 0.6],
        centre_list=[(0.0, 0.0)],
    )

    dataset = dataset.apply_over_sampling(over_sample_size_lp=over_sample_size)

    """
    __Model__
    """
    bulge = af.Model(ag.lp_linear.Sersic)
    disk = af.Model(ag.lp_linear.Exponential)
    bulge.centre = disk.centre

    galaxy = af.Model(ag.Galaxy, redshift=0.5, bulge=bulge, disk=disk)

    model = af.Collection(galaxies=af.Collection(galaxy=galaxy))

    """
    __Search__ 
    """
    search = af.Nautilus(
        path_prefix=Path("imaging"),
        name="modeling",
        unique_tag=dataset_name,
        n_live=100,
        n_batch=50,  # GPU batching and VRAM use explained in `modeling` examples.
        iterations_per_quick_update=100000,
    )

    """
    __Analysis__
    """
    analysis = ag.AnalysisImaging(
        dataset=dataset,
        use_jax=True,  # JAX will use GPUs for acceleration if available, else JAX will use multithreaded CPUs.
    )

    """
    __Model-Fit__
    """
    result = search.fit(model=model, analysis=analysis)

    """
    __Output Folder__
    """
    print(result.info)

    print(result.max_log_likelihood_instance)

    aplt.subplot_galaxies(
        galaxies=result.max_log_likelihood_galaxies, grid=result.grids.lp
    )

    aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)


This small change in how the code is run fixes parallelization issues.

In [ ]:
if __name__ == "__main__":
    fit()
